In [0]:
# Cell 1: setup and helpers
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone
from pyspark.sql import Row

CATALOG = "city_lens"
TARGET = f"{CATALOG}.gold.data_health"
checks = []

def one(sql):
    """Run a SQL query that returns a single value."""
    return spark.sql(sql).first()[0]

def add(source, check_name, value, warn_if, fail_if=None, threshold="", detail=""):
    """Record one check. warn_if / fail_if are functions of value -> bool."""
    if value is None:
        status = "FAIL"
    elif fail_if and fail_if(value):
        status = "FAIL"
    elif warn_if(value):
        status = "WARN"
    else:
        status = "OK"
    checks.append(Row(source=source, check_name=check_name,
                      value=float(value) if value is not None else None,
                      threshold=threshold, status=status, detail=detail))

def hours_since(table, ts_col="ingest_ts"):
    return one(f"""SELECT round((unix_timestamp(current_timestamp()) - unix_timestamp(max({ts_col}))) / 3600, 1)
                   FROM {table}""")

def rows_last_load(table):
    return one(f"SELECT count(*) FROM {table} WHERE ingest_date = (SELECT max(ingest_date) FROM {table})")

In [0]:
# Cell 2: run the checks
B, S, G = f"{CATALOG}.bronze", f"{CATALOG}.silver", f"{CATALOG}.gold"

# Official notices (SRU)
add("official_notices", "hours_since_last_load", hours_since(f"{B}.sru_raw"),
    warn_if=lambda v: v > 26, fail_if=lambda v: v > 50, threshold="warn > 26 h, fail > 50 h")
add("official_notices", "rows_last_load", rows_last_load(f"{B}.sru_raw"),
    warn_if=lambda v: v == 0, threshold="warn = 0",
    detail="0 can be normal on a quiet day, so this is a warning, not a failure")
add("official_notices", "duplicate_ids_in_silver",
    one(f"SELECT count(*) - count(DISTINCT notice_id) FROM {S}.notices"),
    warn_if=lambda v: False, fail_if=lambda v: v > 0, threshold="fail > 0",
    detail="MERGE on notice_id should make this 0")
add("official_notices", "pct_unknown_stage",
    one(f"SELECT round(100 * avg(CASE WHEN stage = 'unknown' THEN 1 ELSE 0 END), 1) FROM {S}.notices"),
    warn_if=lambda v: v > 30, threshold="warn > 30 %",
    detail="title does not say applied / granted / refused")

# Weather (Open-Meteo)
add("weather", "hours_since_last_load", hours_since(f"{B}.weather_raw"),
    warn_if=lambda v: v > 26, fail_if=lambda v: v > 50, threshold="warn > 26 h, fail > 50 h")
add("weather", "forecast_days_ahead",
    one(f"SELECT datediff(max(date_local), current_date()) FROM {S}.weather_hourly"),
    warn_if=lambda v: v < 5, fail_if=lambda v: v < 1, threshold="warn < 5, fail < 1")

# Roadworks (NDW Melvin)
add("ndw_roadworks", "hours_since_last_load", hours_since(f"{B}.ndw_raw"),
    warn_if=lambda v: v > 26, fail_if=lambda v: v > 50, threshold="warn > 26 h, fail > 50 h")
add("ndw_roadworks", "rows_last_load", rows_last_load(f"{B}.ndw_raw"),
    warn_if=lambda v: v < 100, fail_if=lambda v: v == 0, threshold="warn < 100, fail = 0")

last_mod = one(f"SELECT max(source_last_modified) FROM {B}.ndw_raw")     # HTTP date string from NDW
file_age_h = None
if last_mod:
    file_age_h = round((datetime.now(timezone.utc) - parsedate_to_datetime(last_mod)).total_seconds() / 3600, 1)
add("ndw_roadworks", "ndw_file_age_hours", file_age_h,
    warn_if=lambda v: v > 24, fail_if=lambda v: v > 72, threshold="warn > 24 h, fail > 72 h",
    detail=f"Last-Modified header: {last_mod}")

add("ndw_roadworks", "no_coordinates",
    one(f"SELECT count(*) FROM {S}.city_happenings WHERE source = 'ndw_melvin' AND lat IS NULL"),
    warn_if=lambda v: v > 25, threshold="warn > 25", detail="kept in lists, hidden from the map")
add("ndw_roadworks", "approximate_location",
    one(f"SELECT count(*) FROM {S}.city_happenings WHERE lat IS NOT NULL AND NOT location_ok"),
    warn_if=lambda v: v > 50, threshold="warn > 50",
    detail="pin further from the centre than the search box allows")
add("ndw_roadworks", "long_running_over_1_year",
    one(f"SELECT count(*) FROM {S}.road_works WHERE datediff(end_time, start_time) > 365"),
    warn_if=lambda v: False, threshold="info only",
    detail="multi-year projects or placeholder end dates")
add("ndw_roadworks", "duplicate_ids_in_silver",
    one(f"SELECT count(*) - count(DISTINCT situation_id) FROM {S}.road_works"),
    warn_if=lambda v: False, fail_if=lambda v: v > 0, threshold="fail > 0")

# Gold
add("pipeline", "hours_since_gold_refresh",
    hours_since(f"{G}.happenings_enriched", "refreshed_at"),
    warn_if=lambda v: v > 26, fail_if=lambda v: v > 50, threshold="warn > 26 h, fail > 50 h")

print(len(checks), "checks run")

In [0]:
# Cell 3: write gold.data_health and show it
from pyspark.sql import functions as F

health = (
    spark.createDataFrame(checks)
         .withColumn("checked_at", F.current_timestamp())
)

(health.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(TARGET))

display(spark.sql(f"""
    SELECT source, check_name, value, threshold, status, detail
    FROM {TARGET}
    ORDER BY CASE status WHEN 'FAIL' THEN 0 WHEN 'WARN' THEN 1 ELSE 2 END, source, check_name
"""))